# 온통청년 API로 청년정책 데이터 수집
https://www.youthcenter.go.kr/cmnFooter/openapiIntro/oaiDoc

# 1. API 호출하고 DataFrame 변환

In [ ]:
https://www.youthcenter.go.kr/go/ythip/getPlcy

## API_KEY

In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("API_KEY2")
# print(API_KEY)

In [2]:
url = "https://www.youthcenter.go.kr/go/ythip/getPlcy"
all_data = []
page = 1

while True:
    params = {
        "apiKey": API_KEY,
        "pageIndex": page,
        "pageUnit": 100  # 최대치로 설정
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # 실제 데이터 위치
    items = data.get("result", {}).get("youthPolicyList", [])
    
    # 데이터 없으면 종료
    if not items:
        print(f"{page}페이지에서 종료")
        break
    
    all_data.extend(items)
    print(f"{page}페이지 수집 완료 (누적: {len(all_data)}개)")
    
    page += 1
    
    time.sleep(0.3)  # 서버 부하 방지 (중요)

# 데이터프레임 변환
df = pd.DataFrame(all_data)

print("최종 데이터 수:", len(df))
print(df.head())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [1]:
# ──────────────────────────────────────────
# 0. 라이브러리
# ──────────────────────────────────────────
import ast
import os
import re
from datetime import date, datetime
from pathlib import Path

import faiss
import gradio as gr
import numpy as np
import pandas as pd
import tiktoken
from dotenv import load_dotenv
from google import genai
from google.genai import types

/home/ksjin/miniforge3/envs/gemini/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ──────────────────────────────────────────
# 1. 기본 설정
# ──────────────────────────────────────────
BASE_DIR = Path.cwd()

POLICY_PATH    = BASE_DIR / "youth_policy_final.csv"
FINANCIAL_PATH = BASE_DIR / "financial_products.csv"
EMBED_PATH     = BASE_DIR / "policy_embeddings.csv"
FAISS_PATH     = BASE_DIR / "policy_vector_index.faiss"

load_dotenv(BASE_DIR / ".env")
API_KEY = os.getenv("google_api_key")

EMBEDDING_MODEL   = "gemini-embedding-001"
GENERATION_MODEL  = "gemini-2.0-flash"   # [수정] gemini-3-flash-preview → 존재하지 않는 모델명 수정
EMBED_ENCODING    = "cl100k_base"
MAX_CONTEXT_TOKENS = 12000

tokenizer = tiktoken.get_encoding(EMBED_ENCODING)

print("BASE_DIR:", BASE_DIR)
print("POLICY_PATH exists:", POLICY_PATH.exists())
print("FINANCIAL_PATH exists:", FINANCIAL_PATH.exists())
print("API KEY loaded:", bool(API_KEY))

BASE_DIR: /mnt/c/fintech4/08AI_serving/프로젝트
POLICY_PATH exists: True
FINANCIAL_PATH exists: True
API KEY loaded: True


In [3]:
# ──────────────────────────────────────────
# 2. 유틸 함수
# ──────────────────────────────────────────
def safe_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def numeric(val, default: float = 0.0) -> float:
    try:
        if pd.isna(val):
            return default
        return float(val)
    except Exception:
        return default


def compute_age(birth_date: str) -> int:
    try:
        birth = pd.to_datetime(birth_date).date()
    except Exception:
        return 0
    today = date.today()
    return today.year - birth.year - ((today.month, today.day) < (birth.month, birth.day))


# 단순 슬라이싱 대신 주요 광역시도 전부 매핑
_PROVINCE_MAP = {
    "서울": "서울", "경기": "경기", "인천": "인천",
    "부산": "부산", "대구": "대구", "광주": "광주",
    "대전": "대전", "울산": "울산", "세종": "세종",
    "강원": "강원", "충북": "충북", "충남": "충남",
    "전북": "전북", "전남": "전남", "경북": "경북",
    "경남": "경남", "제주": "제주",
}

def province_from_region(region: str) -> str:
    region = safe_str(region)
    for key, prov in _PROVINCE_MAP.items():
        if region.startswith(key):
            return prov
    return region[:2]   # 매핑 실패 시 폴백


def get_client():
    if not API_KEY:
        raise RuntimeError(
            "GOOGLE_API_KEY 또는 google_api_key 환경변수가 필요합니다. "
            ".env 파일에 API 키를 넣어주세요."
        )
    return genai.Client(api_key=API_KEY)

In [4]:
# ──────────────────────────────────────────
# 3. 데이터 로드
# ──────────────────────────────────────────
def load_policy_data() -> pd.DataFrame:
    if not POLICY_PATH.exists():
        raise FileNotFoundError(f"정책 데이터 파일이 없습니다: {POLICY_PATH}")
    return pd.read_csv(POLICY_PATH, encoding="utf-8-sig")


def load_financial_products() -> pd.DataFrame:
    if not FINANCIAL_PATH.exists():
        raise FileNotFoundError(f"금융상품 파일이 없습니다: {FINANCIAL_PATH}")
    return pd.read_csv(FINANCIAL_PATH, encoding="utf-8-sig")


policies_df  = load_policy_data()
financial_df = load_financial_products()
print("정책 데이터:", policies_df.shape)
print("금융상품 데이터:", financial_df.shape)

정책 데이터: (1828, 60)
금융상품 데이터: (1003, 23)


In [5]:
# ──────────────────────────────────────────
# 4. 사용자 객체 생성
# ──────────────────────────────────────────
_SCHOOL_CODE_MAP = {
    "고교 졸업":  "0049004",
    "대학 재학":  "0049005",
    "대졸 예정":  "0049006",
    "대학 졸업":  "0049007",
    "석박사":     "0049008",
    "기타":       "0049009",
}
_JOB_CODE_MAP = {
    "취업":       "0013001",
    "자영업":     "0013002",
    "취업준비중": "0013003",
    "학생":       "0013003",
    "프리랜서":   "0013004",
    "일용근로":   "0013005",
    "예비창업":   "0013006",
    "단기근로":   "0013007",
    "영농종사":   "0013008",
    "기타":       "0013009",
}
_SUPPORT_MAP = {
    "기초생활수급자": "0014003",
    "한부모가정":     "0014004",
    "장애인":         "0014005",
    "농업인":         "0014006",
    "군인":           "0014007",
    "지역인재":       "0014008",
    "여성":           "0014002",
    "중소기업":       "0014001",
}


def build_user_from_form(
    gender, birth_date, education, region,
    monthly_income, annual_income, house_owner, social_support,
    spending_type, marital_status, employment_status, years_of_service,
    household_type, is_sme_employee, asset_total, financial_asset,
    debt_status, housing_type, income_type,
) -> dict:
    age      = compute_age(str(birth_date))
    province = province_from_region(region)

    school_code = _SCHOOL_CODE_MAP.get(education, "0049010")
    job_code    = _JOB_CODE_MAP.get(employment_status, "0013009")

    mrg_code = {"기혼": "0055001", "미혼": "0055002"}.get(marital_status, "0055003")

    # 특화 코드 — '제한없음' 코드(0014010)는 항상 포함
    special_codes: set[str] = {"0014010"}
    if social_support in _SUPPORT_MAP:
        special_codes.add(_SUPPORT_MAP[social_support])
    if gender == "여성":
        special_codes.add("0014002")
    if is_sme_employee == "예":
        special_codes.add("0014001")

    return {
        "user_id":        "INPUT_USER",
        "age":            age,
        "province":       province,
        "region":         region,
        "annual_income":  float(annual_income),
        "monthly_income": float(monthly_income),
        "asset_total":    float(asset_total),
        "financial_asset":float(financial_asset),
        "debt":           debt_status,
        "housing_type":   housing_type,
        "house_owner":    house_owner,
        "spending_type":  spending_type,
        "school_code":    school_code,
        "job_code":       job_code,
        "mrg_code":       mrg_code,
        "special_codes":  special_codes,
        "raw": {
            "gender": gender, "birth_date": str(birth_date),
            "education": education, "region": region,
            "monthly_income": monthly_income, "annual_income": annual_income,
            "house_owner": house_owner, "social_support": social_support,
            "spending_type": spending_type, "marital_status": marital_status,
            "employment_status": employment_status, "years_of_service": years_of_service,
            "household_type": household_type, "is_sme_employee": is_sme_employee,
            "asset_total": asset_total, "financial_asset": financial_asset,
            "debt_status": debt_status, "housing_type": housing_type,
            "income_type": income_type,
        },
    }

In [6]:
# ──────────────────────────────────────────
# 5. Smart Profiling (페르소나 유형화)
# ──────────────────────────────────────────
# [수정] 조건 순서 재정렬 — 소득·자산 기반 유형이 주거 유형보다 먼저 분류되어야 함
def categorize_profile(user: dict) -> str:
    inc  = user["monthly_income"]
    asset = user["asset_total"]
    fin  = user["financial_asset"]
    stype = user["spending_type"]
    hw   = user["house_owner"]
    ht   = user["housing_type"]

    if stype == "투자형" and fin >= 1500:
        return "공격적 자산형"
    if stype in ["절약형", "저축형"] and asset >= 800:
        return "안정 저축형"
    if inc <= 120 and hw == "무주택":
        return "지원금 탐색형"
    if ht in ["월세", "전세"] and hw == "무주택":
        return "주거 안정 우선형"
    return "생활 균형형"


def format_profile_text_from_input(user: dict) -> str:
    persona = categorize_profile(user)
    return (
        f"### 사용자 프로필 요약\n"
        f"- 나이 {user['age']}세 / {user['region']}\n"
        f"- 연소득 {int(user['annual_income'])}만원 / 총자산 {int(user['asset_total'])}만원\n"
        f"- {user['housing_type']} / {user['house_owner']} / 소비성향 {user['spending_type']}\n\n"
        f"**유형화 결과:** {persona}"
    )

In [7]:
# ──────────────────────────────────────────
# 6. 신청기간 처리 (통합)
# ──────────────────────────────────────────
def parse_aply_periods(aply_ymd: str) -> list[tuple[date, date]]:
    """'20260325 ~ 20260408\\N20260209 ~ 20260225' 형식 파싱"""
    text = str(aply_ymd).replace("\\N", "\n").replace("\\n", "\n").strip()
    periods = []
    for start_str, end_str in re.findall(r"(\d{8})\s*~\s*(\d{8})", text):
        try:
            periods.append((
                datetime.strptime(start_str, "%Y%m%d").date(),
                datetime.strptime(end_str, "%Y%m%d").date(),
            ))
        except Exception:
            pass
    return periods


def format_aply_period_for_display(aply_ymd: str) -> str:
    periods = parse_aply_periods(aply_ymd)
    if not periods:
        return "정보 없음"
    return "<br>".join(
        f"{s.year}년 {s.month}월 {s.day}일 ~ {e.year}년 {e.month}월 {e.day}일"
        for s, e in periods
    )


def get_nearest_deadline(aply_ymd: str) -> date | None:
    """진행 중인 기간의 종료일, 없으면 가장 가까운 예정 기간의 종료일"""
    periods = parse_aply_periods(aply_ymd)
    if not periods:
        return None
    today = date.today()
    active = [(s, e) for s, e in periods if s <= today <= e]
    if active:
        return min(active, key=lambda x: x[1])[1]
    future = [(s, e) for s, e in periods if s > today]
    if future:
        return min(future, key=lambda x: x[0])[1]
    return None


def days_to_deadline(policy_row: pd.Series) -> int | None:
    deadline = get_nearest_deadline(safe_str(policy_row.get("aplyYmd", "")))
    if deadline is None:
        return None
    return (deadline - date.today()).days


def is_policy_still_applicable(policy_row: pd.Series) -> bool:
    """신청 가능한 기간이 하나라도 남아 있으면 True"""
    today = date.today()
    for _, end_dt in parse_aply_periods(safe_str(policy_row.get("aplyYmd", ""))):
        if today <= end_dt:
            return True
    return False


def has_valid_application_period(aply_ymd: str) -> bool:
    today = date.today()
    for _, end_dt in parse_aply_periods(aply_ymd):
        if today <= end_dt:
            return True
    return False


def format_date_yyyymmdd(value) -> str:
    s = safe_str(value).split(".")[0]
    if not s or s.lower() == "nan":
        return "정보 없음"
    if re.fullmatch(r"\d{8}", s):
        try:
            dt = datetime.strptime(s, "%Y%m%d").date()
            return f"{dt.year}년 {dt.month}월 {dt.day}일"
        except Exception:
            pass
    return s

In [8]:
# ──────────────────────────────────────────
# 7. 정책 조건 매칭 함수
# ──────────────────────────────────────────
# 설계 원칙:
#   - 정책에 조건값이 아예 없음(빈값/nan) → 제한 없음으로 간주, 통과 but 가산점 없음
#   - 정책에 조건값이 명시됨 → 반드시 사용자 값과 비교, 불일치 시 blocker
#   - 이렇게 해야 조건이 다른 사용자 간 결과가 달라짐

# ── 코드 파싱 공통 헬퍼
_NO_RESTRICT_JOB    = {"0013010"}
_NO_RESTRICT_SCHOOL = {"0049010"}
_NO_RESTRICT_SPCL   = {"0014010"}

def _parse_codes(raw) -> set[str]:
    """콤마 구분 코드 문자열 → set. 빈값/nan이면 빈 set 반환."""
    text = safe_str(raw)
    if not text or text in {"nan", "None"}:
        return set()
    return {c.strip() for c in text.split(",") if c.strip()}


def code_matches(policy_value, user_codes: set[str], no_restrict: set[str]) -> tuple[bool, bool]:
    """
    Returns (passes: bool, has_explicit_condition: bool)
    - passes: 이 정책이 사용자에게 열려 있는지
    - has_explicit_condition: 정책에 조건이 명시되어 있는지 (점수 가산 판단에 사용)
    """
    codes = _parse_codes(policy_value)

    # 정책에 조건 명시 없음 → 전체 대상, 통과 but 조건 없음
    if not codes:
        return True, False

    # 제한없음 코드가 포함 → 전체 대상, 통과 but 조건 없음
    if codes & no_restrict:
        return True, False

    # 조건이 명시됨 → 사용자 코드와 교집합 확인
    return bool(codes & user_codes), True


def age_match(policy_row: pd.Series, age: int) -> tuple[bool, bool]:
    """Returns (passes, has_explicit_condition)"""
    min_age = int(numeric(policy_row.get("sprtTrgtMinAge", 0), 0))
    max_age = int(numeric(policy_row.get("sprtTrgtMaxAge", 0), 0))

    # 연령 조건 없음
    if min_age == 0 and max_age == 0:
        return True, False

    # 조건 명시 → 체크
    passes = True
    if min_age and age < min_age:
        passes = False
    if max_age and age > max_age:
        passes = False
    return passes, True


def income_match(policy_row: pd.Series, annual_income: float) -> tuple[bool, bool]:
    """Returns (passes, has_explicit_condition)"""
    income_code = safe_str(policy_row.get("earnCndSeCd", ""))
    min_amt = numeric(policy_row.get("earnMinAmt", 0), 0)
    max_amt = numeric(policy_row.get("earnMaxAmt", 0), 0)

    # 소득 제한 없음 코드
    if "0043001" in income_code:
        return True, False

    # 소득 범위 조건 자체가 없음
    if not min_amt and not max_amt:
        return True, False

    # 조건 명시 → 체크
    passes = True
    if min_amt and annual_income < min_amt:
        passes = False
    if max_amt and annual_income > max_amt:
        passes = False
    return passes, True


def marriage_matches(policy_value, mrg_code: str) -> tuple[bool, bool]:
    """Returns (passes, has_explicit_condition)"""
    text = safe_str(policy_value)
    if not text or text in {"nan", "None"}:
        return True, False
    # 제한없음 코드
    if "0055003" in text or "55003" in text:
        return True, False
    # 조건 명시 → 체크
    return (mrg_code in text or mrg_code[-4:] in text), True


def region_match(policy_row: pd.Series, province: str) -> tuple[bool, bool]:
    """Returns (passes, has_explicit_condition)"""
    pvsn_group = safe_str(policy_row.get("pvsnInstGroupCd", ""))

    # 전국 코드 명시
    if "0054001" in pvsn_group or "54001" in pvsn_group:
        return True, False

    # 텍스트 기반 지역 추출
    text = " ".join([
        safe_str(policy_row.get("sprvsnInstCdNm", "")),
        safe_str(policy_row.get("operInstCdNm", "")),
        safe_str(policy_row.get("plcyExplnCn", "")),
        safe_str(policy_row.get("addAplyQlfcCndCn", "")),
        safe_str(policy_row.get("zipCd", "")),
    ])

    _ALL_PROVINCES = [
        "서울", "경기", "경기도", "인천", "부산", "대구", "광주", "대전",
        "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주",
    ]
    found_any = any(k in text for k in _ALL_PROVINCES)

    # 지역 키워드 자체가 없으면 전국 정책으로 간주
    if not found_any:
        return True, False

    # 지역 키워드가 있으면 명시적 조건 → 사용자 지역과 비교
    _LOC_MAP = {
        "서울": ["서울"],
        "경기": ["경기", "경기도"],
        "인천": ["인천"],
        "부산": ["부산"], "대구": ["대구"], "광주": ["광주"],
        "대전": ["대전"], "울산": ["울산"], "세종": ["세종"],
        "강원": ["강원"], "충북": ["충북"], "충남": ["충남"],
        "전북": ["전북"], "전남": ["전남"], "경북": ["경북"],
        "경남": ["경남"], "제주": ["제주"],
    }
    passes = any(k in text for k in _LOC_MAP.get(province, [province]))
    return passes, True

In [9]:
# ──────────────────────────────────────────
# 8. 정책 매칭 스코어링
# ──────────────────────────────────────────
# 배점 설계:
#   하드 조건 (불일치 시 즉시 제외):
#     연령 불일치 → 즉시 탈락
#     지역 불일치 → 즉시 탈락
#
#   소프트 조건 (점수화):
#     ┌─────────────┬──────┬──────┐
#     │ 항목         │ 명시 │ 없음 │
#     ├─────────────┼──────┼──────┤
#     │ 소득         │ 20   │  5   │
#     │ 학력         │ 15   │  3   │
#     │ 취업상태      │ 15   │  3   │
#     │ 혼인         │ 10   │  2   │
#     │ 특화대상      │ 10   │  2   │
#     └─────────────┴──────┴──────┘
#     → 조건이 명시되고 사용자와 일치하면 높은 점수,
#       조건이 없으면(= 전체 대상) 낮은 점수 부여
#       → 이렇게 해야 조건을 바꿀수록 결과가 달라짐
#
#   보너스 (사용자 상황과 정책 내용 연계):
#     주거 관련  +8
#     저축/자산  +8
#     청년 특화  +5
#     마감 임박  +5

def compute_match_score(user: dict, policy_row: pd.Series) -> dict:
    score    = 0
    reasons  = []
    blockers = []

    # ══ HARD 조건: 불일치 시 즉시 탈락 ══

    # ① 연령 — 하드 필터
    age_ok, age_explicit = age_match(policy_row, user["age"])
    if not age_ok:
        return {
            "eligible": False, "score": 0,
            "reasons": [], "blockers": ["연령 조건 불일치"],
            "deadline_days": days_to_deadline(policy_row),
        }
    if age_explicit:
        score += 20
        reasons.append(f"연령 조건 일치 (만 {user['age']}세)")
    else:
        score += 5
        reasons.append("연령 제한 없음")

    # ② 지역 — 하드 필터
    region_ok, region_explicit = region_match(policy_row, user["province"])
    if not region_ok:
        return {
            "eligible": False, "score": 0,
            "reasons": [], "blockers": ["지역 조건 불일치"],
            "deadline_days": days_to_deadline(policy_row),
        }
    if region_explicit:
        score += 15
        reasons.append(f"거주 지역({user['province']}) 조건 일치")
    else:
        score += 5
        reasons.append("전국 대상 정책")

    # ══ SOFT 조건: 점수화 ══

    # ③ 소득
    income_ok, income_explicit = income_match(policy_row, user["annual_income"])
    if income_explicit:
        if income_ok:
            score += 20
            reasons.append("소득 조건 명시 — 부합")
        else:
            score -= 15
            blockers.append("소득 조건 불일치")
    else:
        score += 5   # 제한 없음: 소폭만 가산

    # ④ 학력
    school_ok, school_explicit = code_matches(
        policy_row.get("schoolCd"),
        {user["school_code"]},
        _NO_RESTRICT_SCHOOL,
    )
    if school_explicit:
        if school_ok:
            score += 15
            reasons.append("학력 조건 명시 — 부합")
        else:
            score -= 10
            blockers.append("학력 조건 불일치")
    else:
        score += 3

    # ⑤ 취업 상태
    job_ok, job_explicit = code_matches(
        policy_row.get("jobCd"),
        {user["job_code"]},
        _NO_RESTRICT_JOB,
    )
    if job_explicit:
        if job_ok:
            score += 15
            reasons.append("취업 상태 조건 명시 — 부합")
        else:
            score -= 10
            blockers.append("취업 상태 조건 불일치")
    else:
        score += 3

    # ⑥ 혼인
    mrg_ok, mrg_explicit = marriage_matches(policy_row.get("mrgSttsCd"), user["mrg_code"])
    if mrg_explicit:
        if mrg_ok:
            score += 10
            reasons.append("혼인 상태 조건 명시 — 부합")
        else:
            score -= 8
            blockers.append("혼인 상태 조건 불일치")
    else:
        score += 2

    # ⑦ 특화 대상
    spcl_ok, spcl_explicit = code_matches(
        policy_row.get("sbizCd"),
        user["special_codes"],
        _NO_RESTRICT_SPCL,
    )
    if spcl_explicit:
        if spcl_ok:
            score += 10
            reasons.append("특화 대상 조건 명시 — 부합")
        else:
            score -= 8
            blockers.append("특화 조건 불일치")
    else:
        score += 2

    # ══ 보너스 ══

    policy_text = " ".join([
        safe_str(policy_row.get("plcyNm", "")),
        safe_str(policy_row.get("plcyKywdNm", "")),
        safe_str(policy_row.get("plcyExplnCn", "")),
        safe_str(policy_row.get("plcySprtCn", "")),
        safe_str(policy_row.get("lclsfNm", "")),
        safe_str(policy_row.get("mclsfNm", "")),
    ])

    # 주거 상황 연계
    if user["housing_type"] in ["월세", "전세"] and user["house_owner"] == "무주택":
        if any(k in policy_text for k in ["월세", "전세", "주거", "주택", "임차"]):
            score += 8
            reasons.append("주거 상황 맞춤 정책")

    # 저축/자산형성 연계
    if user["spending_type"] in ["절약형", "저축형"]:
        if any(k in policy_text for k in ["저축", "적금", "자산", "계좌", "목돈"]):
            score += 8
            reasons.append("저축/자산형성 관련 정책")

    # 청년 키워드
    if "청년" in policy_text:
        score += 5
        reasons.append("청년 대상 정책")

    # 마감 임박
    d = days_to_deadline(policy_row)
    if d is not None and 0 <= d <= 14:
        score += 5
        reasons.append(f"신청 마감 {d}일 전")

    eligible = len(blockers) == 0
    return {
        "eligible":      eligible,
        "score":         max(score, 0),   # 음수 방지, 상한 없음(정렬용 raw 점수)
        "reasons":       reasons,
        "blockers":      blockers,
        "deadline_days": d,
    }

In [10]:
# ──────────────────────────────────────────
# 9. 정책 추천 빌드
# ──────────────────────────────────────────
def build_recommendations_for_user(user: dict, policies_df: pd.DataFrame) -> pd.DataFrame:
    items = []

    for _, policy in policies_df.iterrows():
        # 승인 상태 체크
        status = safe_str(policy.get("plcyAprvSttsCd", ""))
        if status and status not in {"0044002", "44002", "44002.0"}:
            continue

        # 신청기간 만료 정책 제외
        if not is_policy_still_applicable(policy):
            continue

        result = compute_match_score(user, policy)

        # 하드 조건(연령·지역) 탈락 또는 소프트 조건 누적 감점으로 0점인 경우 제외
        if result["score"] == 0:
            continue

        # eligible(모든 명시 조건 충족)이면 30점, 아니면 50점 이상만 포함
        # → 조건이 다른 사용자끼리 결과가 달라지도록 임계값을 높게 설정
        threshold = 30 if result["eligible"] else 50
        if result["score"] < threshold:
            continue

        items.append({
            "plcyNo":          safe_str(policy.get("plcyNo", "")),
            "plcyNm":          safe_str(policy.get("plcyNm", "")),
            "support_org":     safe_str(policy.get("sprvsnInstCdNm") or policy.get("operInstCdNm")),
            "lclsfNm":         safe_str(policy.get("lclsfNm", "")),
            "mclsfNm":         safe_str(policy.get("mclsfNm", "")),
            "plcySprtCn":      safe_str(policy.get("plcySprtCn", "")),
            "aplyYmd":         safe_str(policy.get("aplyYmd", "")),
            "aplyYmd_display": format_aply_period_for_display(policy.get("aplyYmd", "")),
            "aplyUrlAddr":     safe_str(policy.get("aplyUrlAddr", "")),
            "score":           result["score"],
            "eligible":        result["eligible"],
            "reasons":         "; ".join(result["reasons"][:5]),
            "blockers":        "; ".join(result["blockers"][:3]),
            "deadline_days":   result["deadline_days"],
        })

    rec_df = pd.DataFrame(items)
    if not rec_df.empty:
        # eligible=True 우선, 같은 eligible 내에서 score 내림차순
        rec_df = rec_df.sort_values(
            by=["eligible", "score"],
            ascending=[False, False],
        ).head(30).reset_index(drop=True)

    return rec_df

In [11]:
# ──────────────────────────────────────────
# 10. 골든타임 알림
# ──────────────────────────────────────────
def alert_frame(rec_df: pd.DataFrame) -> pd.DataFrame:
    if rec_df.empty:
        return rec_df

    def classify(days):
        if pd.isna(days):    return "상시/기한 정보 없음"
        if days < 0:         return "마감"
        if days == 0:        return "D-Day"
        if days <= 3:        return "D-3 임박"
        if days <= 7:        return "D-7 안내"
        if days <= 30:       return "이번 달 체크"
        return "일반 안내"

    out = rec_df.copy()
    out["alert_type"] = out["deadline_days"].apply(classify)
    return out.sort_values(["deadline_days", "score"], ascending=[True, False], na_position="last")


def make_alert_table(rec_df: pd.DataFrame, selected_policies: list | None = None, top_n: int = 15) -> pd.DataFrame:
    if rec_df.empty:
        return pd.DataFrame(columns=["정책명", "운영기관", "매칭점수", "남은일수", "알림유형", "신청기간"])

    alerts = alert_frame(rec_df)
    if selected_policies:
        alerts = alerts[alerts["plcyNm"].isin(selected_policies)]

    temp = alerts[["plcyNm", "support_org", "score", "deadline_days", "alert_type", "aplyYmd_display"]].head(top_n).copy()
    temp.columns = ["정책명", "운영기관", "매칭점수", "남은일수", "알림유형", "신청기간"]
    return temp

In [12]:
# ──────────────────────────────────────────
# 11. 금융상품 관련 유틸
# ──────────────────────────────────────────
def format_money(value) -> str:
    try:
        if pd.isna(value):
            return "정보 없음"
        return f"{int(float(value)):,}원"
    except Exception:
        return "정보 없음"


def is_financial_product_active(row: pd.Series) -> bool:
    end_val = safe_str(row.get("종료일", "")).split(".")[0]
    if not end_val or end_val.lower() == "nan":
        return True
    if re.fullmatch(r"\d{8}", end_val):
        try:
            return date.today() <= datetime.strptime(end_val, "%Y%m%d").date()
        except Exception:
            return True
    return True


def extract_join_amount_text(row: pd.Series) -> str:
    max_limit = row.get("최대한도", np.nan)
    if pd.notna(max_limit):
        return format_money(max_limit)
    m = re.search(r"월\s*([0-9,]+)만원\s*이내", safe_str(row.get("유의사항", "")))
    if m:
        return f"월 {m.group(1)}만원 이내"
    return "정보 없음"

In [13]:
# ──────────────────────────────────────────
# 12. 금융상품 추천 (최종 교체본)
# ──────────────────────────────────────────
def _contains_any(text: str, keywords: list[str]) -> bool:
    text = safe_str(text)
    return any(k in text for k in keywords)


def _extract_age_range_from_text(text: str) -> tuple[int | None, int | None]:
    text = safe_str(text)

    # 예: "만 19세 이상", "19세 이상"
    m1 = re.search(r"(?:만\s*)?(\d{1,2})세\s*이상", text)
    min_age = int(m1.group(1)) if m1 else None

    # 예: "34세 이하", "만 34세 이하"
    m2 = re.search(r"(?:만\s*)?(\d{1,2})세\s*이하", text)
    max_age = int(m2.group(1)) if m2 else None

    # 예: "19세~34세", "19세 ~ 34세"
    m3 = re.search(r"(\d{1,2})세\s*[~\-]\s*(\d{1,2})세", text)
    if m3:
        min_age = int(m3.group(1))
        max_age = int(m3.group(2))

    return min_age, max_age


def _financial_target_passes(user: dict, row: pd.Series) -> bool:
    """
    금융상품 가입대상 텍스트 기준 1차 필터
    너무 공격적으로 걸러서 상품이 안 나오는 문제를 막기 위해
    '명백히 불일치'하는 경우만 제외
    """
    target = " ".join([
        safe_str(row.get("가입대상", "")),
        safe_str(row.get("상품명", "")),
        safe_str(row.get("우대조건", "")),
        safe_str(row.get("유의사항", "")),
    ])

    age = user["age"]
    min_age, max_age = _extract_age_range_from_text(target)

    if min_age is not None and age < min_age:
        return False
    if max_age is not None and age > max_age:
        return False

    # 청년 상품인데 고령층 사용자면 제외
    if "청년" in target and age > 39:
        return False

    # 군인 전용
    if "군인" in target and "0014007" not in user["special_codes"]:
        return False

    # 중소기업 특화
    if "중소기업" in target and "0014001" not in user["special_codes"]:
        return False

    # 여성 특화
    if "여성" in target and "0014002" not in user["special_codes"]:
        return False

    return True


def recommend_financial_products(user: dict, savings_df: pd.DataFrame | None = None, top_n: int = 8) -> pd.DataFrame:
    """
    사용자 조건 기반 금융상품 추천
    - 현재 판매 중인 적금/저축 상품만
    - 가입대상 명백 불일치 상품 제외
    - 소비성향/직업/주거상황/사회배려/비대면 여부 등 반영
    - 100점 만점 정규화
    """
    if savings_df is None or savings_df.empty:
        return pd.DataFrame([{
            "회사명": "데이터 없음",
            "상품명": "금융상품 데이터가 없습니다.",
            "추천점수": 0,
            "추천사유": "financial_products.csv를 확인해주세요.",
            "가입대상": "",
            "가입방법": "",
            "최대한도": "",
            "우대조건": "",
            "시작일": "",
            "종료일": "",
        }])

    df = savings_df.copy()

    # 1) 현재 판매 중 상품만
    df = df[df.apply(is_financial_product_active, axis=1)].copy()

    # 2) 적금/저축성 상품만
    df = df[df["상품명"].apply(safe_str).str.contains("적금|저축", na=False)].copy()

    if df.empty:
        return pd.DataFrame([{
            "회사명": "조건 일치 상품 없음",
            "상품명": "현재 가입 가능한 적금 상품이 없습니다.",
            "추천점수": 0,
            "추천사유": "현재 판매 중인 적금 데이터가 없거나 조건에 맞는 상품이 없습니다.",
            "가입대상": "",
            "가입방법": "",
            "최대한도": "",
            "우대조건": "",
            "시작일": "",
            "종료일": "",
        }])

    # 3) 가입대상 명백 불일치 제외
    df = df[df.apply(lambda row: _financial_target_passes(user, row), axis=1)].copy()

    if df.empty:
        return pd.DataFrame([{
            "회사명": "조건 일치 상품 없음",
            "상품명": "가입대상 조건에 맞는 금융상품이 없습니다.",
            "추천점수": 0,
            "추천사유": "현재 사용자 조건과 명백히 맞지 않는 상품은 제외되었습니다.",
            "가입대상": "",
            "가입방법": "",
            "최대한도": "",
            "우대조건": "",
            "시작일": "",
            "종료일": "",
        }])

    scored_rows = []

    for _, row in df.iterrows():
        product_name = safe_str(row.get("상품명", ""))
        company      = safe_str(row.get("회사명", ""))
        join_method  = safe_str(row.get("가입방법", ""))
        target       = safe_str(row.get("가입대상", ""))
        bonus        = safe_str(row.get("우대조건", ""))
        note         = safe_str(row.get("유의사항", ""))
        text_all     = " ".join([product_name, join_method, target, bonus, note])

        raw_score = 0
        reasons = []

        # 기본 적합도
        raw_score += 25
        reasons.append("현재 가입 가능한 적금 상품")

        # 청년 친화도
        if "청년" in text_all:
            raw_score += 15
            reasons.append("청년 대상 또는 청년 친화 상품")

        # 개인 가입 가능
        if _contains_any(target, ["개인", "실명의 개인", "개인고객", "거주자"]):
            raw_score += 10
            reasons.append("개인 가입 가능 상품")

        # 비대면
        if _contains_any(join_method, ["스마트폰", "인터넷", "모바일", "비대면"]):
            raw_score += 12
            reasons.append("비대면 가입 가능")

        # 소비 성향 반영
        if user["spending_type"] in ["절약형", "저축형"]:
            raw_score += 12
            reasons.append("저축 성향과 적합")
        elif user["spending_type"] == "소비형":
            raw_score += 6
            reasons.append("소액 저축으로 시작 가능")
        elif user["spending_type"] == "투자형":
            raw_score += 5
            reasons.append("현금흐름 관리용으로 활용 가능")

        # 소득/납입 부담
        amount_text = extract_join_amount_text(row)
        if user["monthly_income"] <= 200:
            if any(k in amount_text for k in ["300,000", "500,000", "월 30만원", "월 50만원", "50만원"]):
                raw_score += 10
                reasons.append("월 납입 부담이 비교적 낮음")
            else:
                raw_score += 4

        # 주거 상황
        if user["housing_type"] in ["월세", "전세"] and user["house_owner"] == "무주택":
            raw_score += 8
            reasons.append("주거비 관리와 병행 가능한 저축 상품")

        # 재직자 / 급여우대
        if user["job_code"] == "0013001":
            if _contains_any(text_all, ["급여", "카드", "급여이체", "실적"]):
                raw_score += 10
                reasons.append("재직자 우대조건 활용 가능")

        # 사회배려 / 특화
        if "0014001" in user["special_codes"] and "중소기업" in text_all:
            raw_score += 8
            reasons.append("중소기업 재직자에게 유리할 가능성")

        if "0014004" in user["special_codes"] and "한부모" in text_all:
            raw_score += 8
            reasons.append("사회배려자 조건과 연관 가능성")

        if "0014005" in user["special_codes"] and "장애" in text_all:
            raw_score += 8
            reasons.append("사회배려자 조건과 연관 가능성")

        if "0014007" in user["special_codes"] and "군인" in text_all:
            raw_score += 8
            reasons.append("군 관련 우대 가능성")

        # 100점 만점 정규화
        final_score = min(int(raw_score), 100)

        scored_rows.append({
            "회사명": company,
            "상품명": product_name,
            "추천점수": final_score,
            "추천사유": " / ".join(reasons[:4]),
            "가입대상": target if target else "정보 없음",
            "가입방법": join_method if join_method else "정보 없음",
            "최대한도": amount_text,
            "우대조건": bonus if bonus else "정보 없음",
            "시작일": format_date_yyyymmdd(row.get("시작일", "")),
            "종료일": format_date_yyyymmdd(row.get("종료일", "")),
        })

    result_df = pd.DataFrame(scored_rows)

    if result_df.empty:
        return pd.DataFrame([{
            "회사명": "조건 일치 상품 없음",
            "상품명": "추천 가능한 금융상품이 없습니다.",
            "추천점수": 0,
            "추천사유": "조건을 조금 완화해보세요.",
            "가입대상": "",
            "가입방법": "",
            "최대한도": "",
            "우대조건": "",
            "시작일": "",
            "종료일": "",
        }])

    result_df = result_df.sort_values(
        by=["추천점수", "회사명", "상품명"],
        ascending=[False, True, True]
    ).head(top_n).reset_index(drop=True)

    return result_df


# ──────────────────────────────────────────
# 13. 카드/RAG helper
# ──────────────────────────────────────────
def shorten_text(text, max_len=120):
    text = safe_str(text).replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    if len(text) <= max_len:
        return text
    return text[:max_len].rstrip() + "..."


def build_reason_list_html(reasons_text: str) -> str:
    reasons_text = safe_str(reasons_text)
    if not reasons_text:
        return "<li>추천 이유 정보 없음</li>"
    parts = [r.strip() for r in reasons_text.split(";") if r.strip()]
    if not parts:
        return "<li>추천 이유 정보 없음</li>"
    return "".join([f"<li>{p}</li>" for p in parts])


def build_support_content_html(support_text: str):
    text = safe_str(support_text)
    if not text:
        return '<div class="support-empty">지원 내용 정보 없음</div>'

    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace("○", "\n○").replace("•", "\n•").replace("□", "\n□")

    parts = re.split(r"(?:\s*○\s*|\s*•\s*|\s*□\s*)", text)
    parts = [p.strip(" -·") for p in parts if p.strip()]
    if not parts:
        parts = [text]

    cleaned_parts = []
    for p in parts:
        p = re.sub(r"^[ㅇ○•□]+\s*", "", p).strip()
        if p:
            cleaned_parts.append(p)

    if not cleaned_parts:
        return '<div class="support-empty">지원 내용 정보 없음</div>'

    return "<ul class='support-list'>" + "".join([f"<li>{p}</li>" for p in cleaned_parts]) + "</ul>"


def build_policy_cards_html(rec_df: pd.DataFrame, top_n: int = 10) -> str:
    if rec_df.empty:
        return """
        <div class="empty-card">
            <div class="empty-title">추천 가능한 정책이 없습니다.</div>
            <div class="empty-desc">입력 조건을 조금 조정하거나 추가 조건을 다시 확인해보세요.</div>
        </div>
        """

    cards = []
    for _, row in rec_df.head(top_n).iterrows():
        policy_name = safe_str(row.get("plcyNm", ""))
        org = safe_str(row.get("support_org", ""))
        category = f"{safe_str(row.get('lclsfNm', ''))} · {safe_str(row.get('mclsfNm', ''))}".strip(" ·")
        support_html = build_support_content_html(row.get("plcySprtCn", ""))
        aply_ymd_display = safe_str(row.get("aplyYmd_display", ""))
        url = safe_str(row.get("aplyUrlAddr", ""))
        score = int(row.get("score", 0)) if pd.notna(row.get("score", None)) else 0
        reasons_html = build_reason_list_html(row.get("reasons", ""))
        deadline = row.get("deadline_days", None)

        if pd.isna(deadline):
            deadline_badge = '<span class="badge badge-neutral">기한 정보 없음</span>'
        elif deadline < 0:
            deadline_badge = '<span class="badge badge-dead">마감</span>'
        elif deadline == 0:
            deadline_badge = '<span class="badge badge-red">D-Day</span>'
        elif deadline <= 3:
            deadline_badge = f'<span class="badge badge-red">D-{int(deadline)}</span>'
        elif deadline <= 7:
            deadline_badge = f'<span class="badge badge-yellow">D-{int(deadline)}</span>'
        else:
            deadline_badge = f'<span class="badge badge-blue">D-{int(deadline)}</span>'

        if score >= 90:
            score_class = "score-red"
        elif score >= 70:
            score_class = "score-yellow"
        else:
            score_class = "score-blue"

        url_html = f'<a href="{url}" target="_blank" class="card-link">신청 링크 바로가기</a>' if url else ""

        cards.append(f"""
        <div class="policy-card">
            <div class="card-top">
                <div class="card-title-wrap">
                    <div class="policy-title">🏛️ {policy_name}</div>
                    <div class="policy-org">{org}</div>
                </div>
                <div class="policy-score {score_class}">매칭 {score}점</div>
            </div>

            <div class="policy-meta-row">
                <span class="meta-chip">{category if category else '분류 정보 없음'}</span>
                {deadline_badge}
            </div>

            <div class="card-section">
                <div class="section-label">지원 내용</div>
                <div class="section-box support-box">{support_html}</div>
            </div>

            <div class="card-section">
                <div class="section-label">추천 이유</div>
                <div class="section-box reason-box">
                    <ul class="reason-list">{reasons_html}</ul>
                </div>
            </div>

            <div class="card-footer">
                <div class="policy-period"><strong>신청기간</strong><br>{aply_ymd_display if aply_ymd_display else '정보 없음'}</div>
                <div>{url_html}</div>
            </div>
        </div>
        """)

    return "<div class='card-grid'>" + "".join(cards) + "</div>"


def build_financial_reason_list_html(reason_text: str) -> str:
    reason_text = safe_str(reason_text)
    if not reason_text:
        return "<li>추천 사유 정보 없음</li>"
    parts = [p.strip() for p in reason_text.split("/") if p.strip()]
    return "".join([f"<li>{p}</li>" for p in parts[:4]])


def build_financial_cards_html(fin_df: pd.DataFrame) -> str:
    if fin_df.empty:
        return """
        <div class="empty-card">
            <div class="empty-title">추천 가능한 금융상품이 없습니다.</div>
            <div class="empty-desc">조건을 조금 조정하거나 금융상품 데이터를 다시 확인해보세요.</div>
        </div>
        """

    cards = []
    for _, row in fin_df.iterrows():
        company = safe_str(row.get("회사명", ""))
        product = safe_str(row.get("상품명", ""))
        target  = safe_str(row.get("가입대상", ""))
        method  = safe_str(row.get("가입방법", ""))
        limit   = safe_str(row.get("최대한도", ""))
        bonus   = safe_str(row.get("우대조건", ""))
        start   = safe_str(row.get("시작일", ""))
        end     = safe_str(row.get("종료일", ""))
        score   = int(row.get("추천점수", 0)) if pd.notna(row.get("추천점수", None)) else 0
        reasons_html = build_financial_reason_list_html(row.get("추천사유", ""))

        if score >= 90:
            score_class = "score-red"
        elif score >= 70:
            score_class = "score-yellow"
        else:
            score_class = "score-blue"

        cards.append(f"""
        <div class="finance-card">
            <div class="card-top">
                <div class="card-title-wrap">
                    <div class="finance-title">💰 {product}</div>
                    <div class="policy-org">{company}</div>
                </div>
                <div class="policy-score {score_class}">추천 {score}점</div>
            </div>

            <div class="policy-meta-row">
                <span class="finance-chip">가입대상: {target if target else '정보 없음'}</span>
                <span class="finance-chip">가입방법: {method if method else '정보 없음'}</span>
            </div>

            <div class="card-section">
                <div class="section-label">추천 사유</div>
                <div class="section-box finance-reason-box">
                    <ul class="reason-list">{reasons_html}</ul>
                </div>
            </div>

            <div class="card-section">
                <div class="section-label">핵심 정보</div>
                <div class="section-box finance-info-box">
                    <div><strong>최대한도:</strong> {limit if limit else '정보 없음'}</div>
                    <div><strong>우대조건:</strong> {bonus if bonus else '정보 없음'}</div>
                    <div><strong>판매기간:</strong> {start} ~ {end}</div>
                </div>
            </div>
        </div>
        """)

    return "<div class='card-grid'>" + "".join(cards) + "</div>"


def build_alert_cards_html(alerts_df: pd.DataFrame) -> str:
    if alerts_df.empty:
        return """
        <div class="empty-card">
            <div class="empty-title">선택한 정책의 알림 정보가 없습니다.</div>
            <div class="empty-desc">추천 정책 중 신청기간이 남아 있는 정책을 선택해보세요.</div>
        </div>
        """

    cards = []
    for _, row in alerts_df.iterrows():
        title = safe_str(row.get("정책명", ""))
        org   = safe_str(row.get("운영기관", ""))
        score = safe_str(row.get("매칭점수", ""))
        days  = row.get("남은일수", None)
        alert_type = safe_str(row.get("알림유형", ""))
        period = safe_str(row.get("신청기간", ""))

        if pd.isna(days):
            day_text = "기한 정보 없음"
            badge = '<span class="badge badge-neutral">상시</span>'
        elif days < 0:
            day_text = "이미 마감"
            badge = '<span class="badge badge-dead">마감</span>'
        elif days == 0:
            day_text = "오늘 마감"
            badge = '<span class="badge badge-red">D-Day</span>'
        elif days <= 3:
            day_text = f"{int(days)}일 남음"
            badge = f'<span class="badge badge-red">D-{int(days)}</span>'
        elif days <= 7:
            day_text = f"{int(days)}일 남음"
            badge = f'<span class="badge badge-yellow">D-{int(days)}</span>'
        else:
            day_text = f"{int(days)}일 남음"
            badge = f'<span class="badge badge-blue">D-{int(days)}</span>'

        cards.append(f"""
        <div class="alert-card">
            <div class="card-top">
                <div class="card-title-wrap">
                    <div class="policy-title">{title}</div>
                    <div class="policy-org">{org}</div>
                </div>
                {badge}
            </div>
            <div class="alert-info"><strong>알림 유형</strong> {alert_type}</div>
            <div class="alert-info"><strong>남은 기간</strong> {day_text}</div>
            <div class="alert-info"><strong>매칭 점수</strong> {score}</div>
            <div class="alert-info"><strong>신청기간</strong><br>{period if period else '정보 없음'}</div>
        </div>
        """)

    return "<div class='card-grid'>" + "".join(cards) + "</div>"


def build_policy_context(rec_df: pd.DataFrame, max_items: int = 8) -> str:
    if rec_df is None or rec_df.empty:
        return "추천된 정책이 없습니다."

    lines = []
    for _, row in rec_df.head(max_items).iterrows():
        lines.append(
            f"정책명: {safe_str(row.get('plcyNm',''))}\n"
            f"운영기관: {safe_str(row.get('support_org',''))}\n"
            f"지원내용: {safe_str(row.get('plcySprtCn',''))}\n"
            f"추천이유: {safe_str(row.get('reasons',''))}\n"
            f"신청기간: {safe_str(row.get('aplyYmd_display',''))}"
        )
    return "\n\n---\n\n".join(lines)


def build_financial_context(fin_df: pd.DataFrame, max_items: int = 5) -> str:
    if fin_df is None or fin_df.empty:
        return "추천된 금융상품이 없습니다."

    lines = []
    for _, row in fin_df.head(max_items).iterrows():
        lines.append(
            f"금융상품명: {safe_str(row.get('상품명',''))}\n"
            f"금융회사: {safe_str(row.get('회사명',''))}\n"
            f"추천점수: {safe_str(row.get('추천점수',''))}\n"
            f"추천사유: {safe_str(row.get('추천사유',''))}\n"
            f"가입대상: {safe_str(row.get('가입대상',''))}\n"
            f"가입방법: {safe_str(row.get('가입방법',''))}\n"
            f"최대한도: {safe_str(row.get('최대한도',''))}\n"
            f"우대조건: {safe_str(row.get('우대조건',''))}\n"
            f"판매기간: {safe_str(row.get('시작일',''))} ~ {safe_str(row.get('종료일',''))}"
        )
    return "\n\n---\n\n".join(lines)


def build_personalized_prompt(question: str, user: dict, rec_df: pd.DataFrame, fin_df: pd.DataFrame) -> str:
    policy_context    = build_policy_context(rec_df)
    financial_context = build_financial_context(fin_df)

    return f"""
당신은 청년 정책과 금융상품을 함께 안내하는 청년 AI 상담 도우미입니다.
반드시 아래 사용자 정보, 추천 정책, 추천 금융상품만 근거로 답변하세요.
추측하지 말고, 정보가 없으면 없다고 말하세요.
답변은 한국어로 간결하지만 실용적으로 작성하세요.

[사용자 정보]
- 나이: {user['age']}
- 지역: {user['region']}
- 연소득: {int(user['annual_income'])}만원
- 총자산: {int(user['asset_total'])}만원
- 금융자산: {int(user['financial_asset'])}만원
- 주거형태: {user['housing_type']}
- 주택보유: {user['house_owner']}
- 소비성향: {user['spending_type']}
- 프로필 유형: {categorize_profile(user)}

[추천 정책 문맥]
{policy_context}

[추천 금융상품 문맥]
{financial_context}

[사용자 질문]
{question}

[답변 형식]
1. 핵심 답변
2. 관련 정책 또는 금융상품 1~3개
3. 사용자가 바로 확인할 포인트
""".strip()


def answer_question(question: str, user: dict, rec_df: pd.DataFrame, fin_df: pd.DataFrame) -> str:
    if rec_df.empty and (fin_df is None or fin_df.empty):
        return "현재 조건에 맞는 추천 정책이나 금융상품이 없어 답변할 수 없습니다. 먼저 조건을 입력하고 추천을 받아보세요."

    client = get_client()
    prompt = build_personalized_prompt(question, user, rec_df, fin_df)

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
    )
    return response.text.strip()


# ──────────────────────────────────────────
# 14. 실행 함수
# ──────────────────────────────────────────
def run_profile_and_recommend(
    gender, birth_date, education, region,
    monthly_income, annual_income, house_owner, social_support,
    spending_type, marital_status, employment_status, years_of_service,
    household_type, is_sme_employee, asset_total, financial_asset,
    debt_status, housing_type, income_type,
    new_policy_alert_opt_in
):
    user = build_user_from_form(
        gender, birth_date, education, region,
        monthly_income, annual_income, house_owner, social_support,
        spending_type, marital_status, employment_status, years_of_service,
        household_type, is_sme_employee, asset_total, financial_asset,
        debt_status, housing_type, income_type,
    )

    rec_df = build_recommendations_for_user(user, policies_df)
    fin_df = recommend_financial_products(user, financial_df, top_n=8)

    profile_md      = format_profile_text_from_input(user)
    card_html       = build_policy_cards_html(rec_df, top_n=10)
    fin_cards_html  = build_financial_cards_html(fin_df)

    selectable_alert_df = rec_df[rec_df["aplyYmd"].apply(has_valid_application_period)].copy() if not rec_df.empty else pd.DataFrame()
    selectable = selectable_alert_df["plcyNm"].tolist() if not selectable_alert_df.empty else []

    if selectable_alert_df.empty:
        alert_intro = "추천 정책 중 신청기간이 남아 있는 정책이 없어 현재 알림 선택이 어렵습니다."
    else:
        alert_intro = "추천 정책 중 신청기간이 남아 있는 정책만 골든타임 알림 대상으로 선택할 수 있습니다."

    new_status = (
        """
**신규 정책 알림 신청 상태: ON**

내 조건에 맞는 신규 정책이 데이터에 추가되면 알림 대상으로 탐지하도록 설정한 상태입니다.  
현재는 UI 프로토타입 단계이며, 실제 자동 알림 기능은 추후 DB 저장 + 주기적 정책 비교 로직과 연결됩니다.
"""
        if new_policy_alert_opt_in else
        """
**신규 정책 알림 신청 상태: OFF**

지금은 사용자가 직접 선택한 정책의 골든타임 알림만 확인하는 상태입니다.
"""
    )

    empty_alert_html = build_alert_cards_html(pd.DataFrame(columns=["정책명", "운영기관", "매칭점수", "남은일수", "알림유형", "신청기간"]))

    return (
        profile_md,
        card_html,
        fin_cards_html,
        gr.update(choices=selectable, value=[], interactive=True),
        rec_df.to_dict("records"),
        alert_intro,
        empty_alert_html,
        new_status,
    )


def update_selected_alerts(selected_policy_names: list, rec_records: list):
    empty_html = build_alert_cards_html(pd.DataFrame(columns=["정책명", "운영기관", "매칭점수", "남은일수", "알림유형", "신청기간"]))
    if not rec_records:
        return "먼저 정책 추천을 받아주세요.", empty_html
    if not selected_policy_names:
        return "알림 받고 싶은 정책을 하나 이상 선택해 주세요.", empty_html

    rec_df    = pd.DataFrame(rec_records)
    rec_df    = rec_df[rec_df["aplyYmd"].apply(has_valid_application_period)].copy()
    alerts_df = make_alert_table(rec_df, selected_policy_names, top_n=15)
    alert_html = build_alert_cards_html(alerts_df)

    if alerts_df.empty:
        return "선택한 정책에 대해 표시할 알림 정보가 없습니다.", alert_html

    valid        = alerts_df["남은일수"].dropna()
    urgent_count = int(((valid >= 0) & (valid <= 7)).sum())
    summary = (
        f"선택한 정책 중 **마감 7일 이내 정책이 {urgent_count}건** 있습니다."
        if urgent_count > 0 else
        "선택한 정책 중 **7일 이내 마감 정책은 없습니다.**"
    )
    return summary, alert_html


def chat_with_form_user(
    gender, birth_date, education, region,
    monthly_income, annual_income, house_owner, social_support,
    spending_type, marital_status, employment_status, years_of_service,
    household_type, is_sme_employee, asset_total, financial_asset,
    debt_status, housing_type, income_type,
    new_policy_alert_opt_in,
    question: str, history: list,
):
    if history is None:
        history = []
    if not question or not str(question).strip():
        return history, ""

    user = build_user_from_form(
        gender, birth_date, education, region,
        monthly_income, annual_income, house_owner, social_support,
        spending_type, marital_status, employment_status, years_of_service,
        household_type, is_sme_employee, asset_total, financial_asset,
        debt_status, housing_type, income_type,
    )
    rec_df = build_recommendations_for_user(user, policies_df)
    fin_df = recommend_financial_products(user, financial_df, top_n=5)

    try:
        answer = answer_question(question, user, rec_df, fin_df)
    except Exception as e:
        answer = f"질의응답 중 오류 발생: {e}"

    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})
    return history, ""


# ──────────────────────────────────────────
# 15. 최종 UI (현재 최신 카드형 UI 반영)
# ──────────────────────────────────────────
CSS = """
.gradio-container {
    max-width: 1280px !important;
    margin: 0 auto !important;
    padding-top: 12px !important;
    background: #f7fbff;
}
.hero-box {
    background: #ffffff;
    border: 1px solid #dbeafe;
    border-radius: 24px;
    padding: 28px 24px;
    margin-bottom: 18px;
    box-shadow: 0 6px 18px rgba(37,99,235,.05);
}
.section-card {
    background: #ffffff;
    border: 1px solid #dde7f2;
    border-radius: 22px;
    padding: 20px;
    box-shadow: 0 8px 24px rgba(15,23,42,.04);
}
.section-title {
    font-size: 20px;
    font-weight: 800;
    margin-bottom: 6px;
    color: #0f172a;
}
.section-desc {
    font-size: 14px;
    color: #475569;
    margin-bottom: 14px;
    line-height: 1.5;
}
.sub-card {
    background: #fcfdff;
    border: 1px solid #e2e8f0;
    border-radius: 18px;
    padding: 14px 14px 8px 14px;
}
.result-title {
    font-size: 18px;
    font-weight: 800;
    margin-bottom: 6px;
    color: #111827;
}
.result-desc {
    font-size: 13px;
    color: #6b7280;
    margin-bottom: 12px;
}
.card-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(330px, 1fr));
    gap: 16px;
    margin-top: 8px;
}
.policy-card, .finance-card, .alert-card {
    border-radius: 20px;
    padding: 18px;
}
.policy-card {
    background: #F8FBFF;
    border: 1px solid #BFDBFE;
    box-shadow: 0 8px 20px rgba(59,130,246,.06);
    transition: all .2s ease;
}
.policy-card:hover {
    transform: translateY(-3px);
    box-shadow: 0 12px 26px rgba(59,130,246,.12);
}
.finance-card {
    background: #FFFDF7;
    border: 1px solid #FCD34D;
    box-shadow: 0 8px 20px rgba(245,158,11,.08);
    transition: all .2s ease;
}
.finance-card:hover {
    transform: translateY(-3px);
    box-shadow: 0 12px 26px rgba(245,158,11,.12);
}
.alert-card {
    background: #ffffff;
    border: 1px solid #fde68a;
    box-shadow: 0 6px 18px rgba(245,158,11,.08);
}
.card-top {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    gap: 12px;
    margin-bottom: 12px;
}
.card-title-wrap { min-width: 0; }
.policy-title, .finance-title {
    font-size: 18px;
    font-weight: 800;
    line-height: 1.35;
    word-break: keep-all;
}
.policy-title { color: #1D4ED8; }
.finance-title { color: #C2410C; }
.policy-org {
    font-size: 13px;
    color: #64748b;
    margin-top: 4px;
}
.policy-score {
    border-radius: 14px;
    padding: 7px 11px;
    font-size: 13px;
    font-weight: 800;
    white-space: nowrap;
    border: 1px solid transparent;
}
.score-blue   { background:#dbeafe; color:#1d4ed8; border-color:#bfdbfe; }
.score-yellow { background:#fef3c7; color:#b45309; border-color:#fde68a; }
.score-red    { background:#fee2e2; color:#b91c1c; border-color:#fecaca; }
.policy-meta-row {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-bottom: 12px;
}
.meta-chip {
    background: #DBEAFE;
    color: #1E40AF;
    padding: 6px 10px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 700;
    border: 1px solid #93C5FD;
}
.finance-chip {
    background: #FEF3C7;
    color: #9A3412;
    padding: 6px 12px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 700;
    border: 1px solid #FCD34D;
    display: inline-block;
    margin-right: 6px;
}
.badge {
    padding: 6px 10px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 800;
}
.badge-red     { background:#fee2e2; color:#b91c1c; }
.badge-yellow  { background:#fef3c7; color:#b45309; }
.badge-blue    { background:#dbeafe; color:#1d4ed8; }
.badge-neutral { background:#e2e8f0; color:#475569; }
.badge-dead    { background:#e5e7eb; color:#6b7280; }
.card-section { margin-bottom: 12px; }
.section-label {
    font-size: 12px;
    font-weight: 800;
    color: #475569;
    margin-bottom: 6px;
    text-transform: uppercase;
    letter-spacing: .03em;
}
.section-box {
    border-radius: 14px;
    padding: 12px 14px;
    font-size: 14px;
    line-height: 1.55;
    color: #1f2937;
}
.support-box {
    background: #EFF6FF;
    border: 1px solid #BFDBFE;
}
.reason-box {
    background: #EFF6FF;
    border: 1px solid #93C5FD;
}
.finance-reason-box {
    background: #FFF7ED;
    border: 1px solid #FDBA74;
}
.finance-info-box {
    background: #FEF3C7;
    border: 1px solid #FCD34D;
}
.support-list, .reason-list {
    margin: 0;
    padding-left: 18px;
}
.support-list li, .reason-list li {
    margin-bottom: 6px;
    line-height: 1.5;
}
.support-empty { color:#64748b; }
.card-footer {
    display:flex;
    flex-direction:column;
    gap:10px;
    margin-top:12px;
    padding-top:12px;
    border-top:1px solid #e5e7eb;
}
.policy-period {
    font-size: 13px;
    color: #374151;
    line-height: 1.5;
}
.card-link {
    display:inline-block;
    color:#2563eb;
    text-decoration:none;
    font-weight:800;
    font-size:13px;
}
.card-link:hover { text-decoration:underline; }
.alert-info {
    font-size: 14px;
    color: #334155;
    margin-bottom: 6px;
    line-height: 1.5;
}
.empty-card {
    border:1px dashed #cbd5e1;
    background:#f8fafc;
    border-radius:18px;
    padding:24px;
    text-align:center;
}
.empty-title {
    font-size:17px;
    font-weight:700;
    color:#334155;
    margin-bottom:6px;
}
.empty-desc {
    font-size:14px;
    color:#64748b;
}
.footer-note {
    color:#64748b;
    font-size:13px;
}
.chat-row { align-items:end !important; }
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="violet", neutral_hue="slate"),
    css=CSS,
) as demo:

    rec_state = gr.State([])

    with gr.Column(elem_classes=["hero-box"]):
        gr.Markdown("""
        # 청년 맞춤 금융·정책 추천 서비스
        사용자가 직접 조건을 입력하면 **지금 신청 가능한 청년 정책**과 **맞춤 금융상품**을 추천하고,
        **알림 받고 싶은 정책만 선택해서 D-Day 관리**할 수 있도록 구성한 서비스입니다.
        """)

    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            with gr.Column(elem_classes=["section-card"]):
                gr.Markdown('<div class="section-title">1. 사용자 정보 입력</div>')
                gr.Markdown('<div class="section-desc">기본 정보는 필수 입력이며, 추가 조건은 선택 입력입니다.</div>')

                with gr.Column(elem_classes=["sub-card"]):
                    gr.Markdown("### 기본 정보")
                    with gr.Row():
                        gender     = gr.Radio(["남성", "여성"], label="성별", value="여성")
                        birth_date = gr.Textbox(label="생년월일", value="2000-01-01", placeholder="YYYY-MM-DD")
                        education  = gr.Dropdown(
                            ["고교 졸업", "대학 재학", "대졸 예정", "대학 졸업", "석박사", "기타"],
                            label="학력", value="대학 졸업",
                        )
                    region = gr.Textbox(label="지역(시/군/구)", value="서울관악구", placeholder="예: 서울관악구")
                    with gr.Row():
                        monthly_income = gr.Number(label="월 소득(만원)", value=200)
                        annual_income  = gr.Number(label="연 소득(만원)", value=2400)
                    with gr.Row():
                        house_owner    = gr.Radio(["무주택", "주택보유"], label="주택 여부", value="무주택")
                        social_support = gr.Dropdown(
                            ["없음", "기초생활수급자", "한부모가정", "장애인", "농업인", "군인", "지역인재", "여성", "중소기업"],
                            label="사회배려자 대상 여부", value="없음",
                        )

                with gr.Accordion("추가 조건 입력 (선택)", open=False):
                    with gr.Column(elem_classes=["sub-card"]):
                        gr.Markdown("### 추가 조건")
                        with gr.Row():
                            spending_type = gr.Dropdown(["절약형", "소비형", "저축형", "투자형"], label="소비 성향", value="저축형")
                            marital_status = gr.Radio(["미혼", "기혼"], label="결혼 여부", value="미혼")
                            employment_status = gr.Dropdown(
                                ["취업", "취업준비중", "학생", "프리랜서", "자영업", "일용근로", "예비창업", "단기근로", "영농종사", "기타"],
                                label="고용 상태", value="취업준비중",
                            )
                        with gr.Row():
                            years_of_service = gr.Dropdown(["0년", "1년 미만", "1~3년", "3~5년", "5년 이상"], label="근속 기간", value="0년")
                            household_type   = gr.Dropdown(["1인 가구", "부모 동거", "부부", "다자녀"], label="가구 형태", value="1인 가구")
                            is_sme_employee  = gr.Radio(["예", "아니오"], label="중소기업 재직 여부", value="아니오")
                        with gr.Row():
                            asset_total     = gr.Number(label="총 자산 규모(만원)", value=500)
                            financial_asset = gr.Number(label="금융자산 규모(만원)", value=200)
                            debt_status     = gr.Dropdown(["없음", "학자금대출", "신용대출", "기타"], label="부채 여부", value="없음")
                        with gr.Row():
                            housing_type = gr.Dropdown(["월세", "전세", "자가", "부모동거", "기숙사"], label="주거 형태", value="월세")
                            income_type  = gr.Dropdown(["근로소득", "사업소득", "무소득", "기타"], label="소득 유형", value="무소득")

                new_policy_alert_opt_in = gr.Checkbox(
                    label="내 조건에 맞는 신규 정책이 들어오면 알림받기",
                    value=False
                )

                with gr.Row():
                    submit_btn = gr.Button("입력 완료 · 추천 받기", variant="primary", size="lg")
                    reset_btn  = gr.Button("입력 초기화", variant="secondary")

        with gr.Column(scale=4):
            with gr.Column(elem_classes=["section-card"]):
                gr.Markdown('<div class="section-title">2. 사용자 요약</div>')
                gr.Markdown('<div class="section-desc">입력 완료 버튼을 누르면 사용자 조건을 분석해 금융 MBTI와 추천 방향을 보여줍니다.</div>')
                profile_md = gr.Markdown("아직 사용자 분석 전입니다.")

            with gr.Column(elem_classes=["section-card"]):
                gr.Markdown('<div class="section-title">3. 신규 정책 알림 상태</div>')
                gr.Markdown('<div class="section-desc">사용자 조건에 맞는 신규 정책이 추가될 때 알림받는 기능의 UI 프로토타입입니다.</div>')
                new_policy_alert_status_md = gr.Markdown("신규 정책 알림 신청 상태: OFF")

    with gr.Tabs():
        with gr.Tab("추천 정책"):
            with gr.Column(elem_classes=["section-card"]):
                gr.Markdown('<div class="result-title">현재 신청 가능한 정책</div>')
                gr.Markdown('<div class="result-desc">입력된 조건을 바탕으로 현재 신청 가능성이 높은 정책을 카드형으로 보여줍니다.</div>')
                preview_cards = gr.HTML(label="추천 정책 카드")

        with gr.Tab("추천 금융상품"):
            with gr.Column(elem_classes=["section-card"]):
                gr.Markdown('<div class="result-title">맞춤 금융상품 추천</div>')
                gr.Markdown('<div class="result-desc">사용자의 소비 성향, 소득 수준, 주거 형태를 바탕으로 지금 가입 가능한 적금·저축성 금융상품을 카드형으로 추천합니다.</div>')
                finance_cards = gr.HTML(label="추천 금융상품 카드")

        with gr.Tab("D-Day 알림"):
            with gr.Column(elem_classes=["section-card"]):
                gr.Markdown('<div class="result-title">신청 골든타임 알림</div>')
                gr.Markdown('<div class="result-desc">신청기간이 남아 있는 정책만 선택해서 D-Day 알림을 확인할 수 있습니다.</div>')

                alert_selector = gr.Dropdown(
                    choices=[],
                    value=[],
                    multiselect=True,
                    label="알림 받고 싶은 정책 선택",
                )

                alert_summary_md = gr.Markdown("아직 선택된 정책이 없습니다.")
                alert_cards = gr.HTML(label="선택 정책 알림 카드")

        with gr.Tab("청년 AI helper"):
            with gr.Column(elem_classes=["section-card"]):
                gr.Markdown('<div class="result-title">청년 AI 상담</div>')
                gr.Markdown('<div class="result-desc">추천된 정책과 금융상품을 기반으로 개인 상황에 맞는 설명을 제공합니다.</div>')

                chatbot = gr.Chatbot(type="messages", label="청년 AI 상담 챗봇", height=420)

                with gr.Row(elem_classes=["chat-row"]):
                    chat_input = gr.Textbox(
                        label="질문 입력",
                        placeholder="예: 나한테 맞는 청년 정책과 금융상품 추천해줘",
                        scale=9,
                    )
                    send_btn = gr.Button("질문하기", variant="primary", scale=1, min_width=120)

                clear_btn = gr.Button("대화 초기화", variant="secondary")

    gr.Markdown('<div class="footer-note">※ 신규 정책 자동 알림은 현재 UI 프로토타입 단계이며, 실제 서비스에서는 사용자 조건 저장과 정책 데이터 주기 업데이트가 추가로 필요합니다.</div>')

    common_inputs = [
        gender, birth_date, education, region,
        monthly_income, annual_income, house_owner, social_support,
        spending_type, marital_status, employment_status, years_of_service,
        household_type, is_sme_employee, asset_total, financial_asset,
        debt_status, housing_type, income_type,
    ]

    submit_btn.click(
        fn=run_profile_and_recommend,
        inputs=common_inputs + [new_policy_alert_opt_in],
        outputs=[
            profile_md,
            preview_cards,
            finance_cards,
            alert_selector,
            rec_state,
            alert_summary_md,
            alert_cards,
            new_policy_alert_status_md,
        ],
    )

    alert_selector.change(
        fn=update_selected_alerts,
        inputs=[alert_selector, rec_state],
        outputs=[alert_summary_md, alert_cards],
    )

    send_btn.click(
        fn=chat_with_form_user,
        inputs=common_inputs + [new_policy_alert_opt_in, chat_input, chatbot],
        outputs=[chatbot, chat_input],
    )

    chat_input.submit(
        fn=chat_with_form_user,
        inputs=common_inputs + [new_policy_alert_opt_in, chat_input, chatbot],
        outputs=[chatbot, chat_input],
    )

    clear_btn.click(
        fn=lambda: [],
        inputs=None,
        outputs=chatbot,
    )

    reset_btn.click(
        fn=lambda: (
            "여성", "2000-01-01", "대학 졸업", "서울관악구",
            200, 2400, "무주택", "없음",
            "저축형", "미혼", "취업준비중", "0년",
            "1인 가구", "아니오", 500, 200,
            "없음", "월세", "무소득",
            False,
            "아직 사용자 분석 전입니다.",
            build_policy_cards_html(pd.DataFrame()),
            build_financial_cards_html(pd.DataFrame()),
            gr.update(choices=[], value=[]),
            [],
            "아직 선택된 정책이 없습니다.",
            build_alert_cards_html(pd.DataFrame()),
            "신규 정책 알림 신청 상태: OFF",
            [],
        ),
        inputs=None,
        outputs=[
            gender, birth_date, education, region,
            monthly_income, annual_income, house_owner, social_support,
            spending_type, marital_status, employment_status, years_of_service,
            household_type, is_sme_employee, asset_total, financial_asset,
            debt_status, housing_type, income_type,
            new_policy_alert_opt_in,
            profile_md,
            preview_cards,
            finance_cards,
            alert_selector,
            rec_state,
            alert_summary_md,
            alert_cards,
            new_policy_alert_status_md,
            chatbot,
        ],
    )

demo.launch(inline=False, share=False)

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.
